# Pynapple event analysis for lifetime data

Run `01_lifetime_qc.ipynb` first. This notebook converts the aligned lifetime, running, and behavior data into Pynapple objects and demonstrates Ensure-aligned analysis.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap
from iflip3 import process_aligned_session, session_paths

If Pynapple is unavailable, run `python -m pip install -e ".[events]"` from an Anaconda Prompt in the repository folder.

## Session configuration

In [ ]:
MOUSE = "MOUSE_ID"
DATE = "YYMMDD"
RUN = 1
DATA_ROOT = Path("Z:/")
BACKGROUND_FILE = Path(r"Z:\FLIM FLIP\backgrounds\matched_background.iFLiP3")
AFTERPULSE_RATIO = 0.03
WINDOW_SECONDS = (-20.0, 40.0)

In [ ]:
if MOUSE == "MOUSE_ID" or DATE == "YYMMDD":
    raise ValueError("Set MOUSE and DATE in the configuration cell first.")
paths = session_paths(MOUSE, DATE, RUN, data_root=DATA_ROOT)
paths.validate()
if not BACKGROUND_FILE.is_file():
    raise FileNotFoundError(f"Background file not found: {BACKGROUND_FILE}")
session = process_aligned_session(
    paths.iflip, paths.nidaq, BACKGROUND_FILE, running_path=paths.running,
    afterpulse_ratio=AFTERPULSE_RATIO,
)
data = session.to_pynapple()
data

The dictionary contains MPET and raw intensity as `Tsd`, event onsets as `Ts`, event durations as `IntervalSet`, and running speed as `Tsd`.

## Aligned session overview

In [ ]:
time = session.lifetime_time_nidaq
baseline = time < time[0] + min(50.0, np.ptp(time) / 6)
delta_mpet = session.mpet_ns - np.nanmedian(session.mpet_ns[baseline])
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
axes[0].plot(time, delta_mpet); axes[0].set_ylabel("Delta MPET (ns)")
axes[1].eventplot(session.lick_times_nidaq, colors="black"); axes[1].set_ylabel("Licks")
axes[2].plot(session.running_time_nidaq, session.running_speed, color="tab:green")
axes[2].set(xlabel="NI-DAQ time (s)", ylabel="Running speed")
for onset, offset in zip(session.ensure_pulses.onset_times, session.ensure_pulses.offset_times, strict=True):
    for ax in axes: ax.axvspan(onset, offset, color="tab:orange", alpha=0.2)
plt.show()

## Ensure-aligned MPET

Each column is one Ensure event. Subtracting each trial's pre-event mean emphasizes within-trial lifetime changes.

In [ ]:
mpet_trials = nap.compute_perievent(data["mpet"], data["ensure"], window=WINDOW_SECONDS, time_unit="s")
relative_time = np.asarray(mpet_trials.t)
values = np.asarray(mpet_trials)
if values.ndim == 1: values = values[:, None]
delta_trials = values - np.nanmean(values[relative_time < 0], axis=0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(relative_time, delta_trials, color="0.75", alpha=0.8)
ax.plot(relative_time, np.nanmean(delta_trials, axis=1), color="tab:blue", lw=2)
ax.axvline(0, color="tab:orange"); ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="Time from Ensure onset (s)", ylabel="Delta MPET (ns)")
plt.show()

## Ensure-aligned running

In [ ]:
running_trials = nap.compute_perievent(data["running_speed"], data["ensure"], window=WINDOW_SECONDS, time_unit="s")
running_time = np.asarray(running_trials.t)
running_values = np.asarray(running_trials)
if running_values.ndim == 1: running_values = running_values[:, None]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(running_time, running_values, color="0.75", alpha=0.8)
ax.plot(running_time, np.nanmean(running_values, axis=1), color="tab:green", lw=2)
ax.axvline(0, color="tab:orange")
ax.set(xlabel="Time from Ensure onset (s)", ylabel="Running speed")
plt.show()

## Ensure-aligned lick timestamps

The returned Pynapple object contains lick times relative to each Ensure event and can be used for rasters or lick-rate histograms.

In [ ]:
licks_around_ensure = nap.compute_perievent(
    data["licks"], data["ensure"], window=(-5.0, 10.0), time_unit="s"
)
licks_around_ensure

### Interpretation checklist

- Inspect individual trials before interpreting the mean.
- Define and report the baseline window before comparing conditions.
- Use intensity as QC rather than the primary response variable.
- Preserve the background filename, afterpulse ratio, and synchronization diagnostics.